# 🚀 Asistente de Documentación IEEE con IA

Este cuaderno automatiza el flujo de trabajo para la creación de artículos científicos bajo el estándar **IEEE**. Utiliza una combinación de **Ingeniería de Prompts**, **Procesamiento de Lenguaje Natural** y **Automatización de Archivos** para transformar una idea en un entorno LaTeX profesional.

## 🔄 Flujo de Trabajo:
1.  **Generación de Metaprompt**: Define la estructura técnica, bibliografía BibTeX y mapa de uso.
2.  **Gestión de Plan**: Captura y guarda el plan de investigación en formato Markdown.
3.  **Procesamiento de Documentos**: Genera automáticamente la estructura de carpetas, archivos `.tex`, archivo `.bib` y un `main.tex` configurado con normas IEEE y enlaces ORCID.
4.  **Redacción Asistida**: Proporciona prompts específicos por sección para alimentar a una IA y un procesador directo que inyecta el contenido de vuelta al proyecto. Acá se crean automáticamente *placeholders* (espacios reservados) para figuras y diagramas técnicos.
5.  **Compresión y descarga**: Se obtienen los outputs del proceso.
6. **Datos de registro**: Se crea el prompt para generar los datos de regitro del proyecto en el sistema de gestion de informacion de la organización para la cual fue creada la investigación.
7. **Limpieza del entorno**: Borrado del directorio `workflow/` y `outputs.zip` (Opción normamente desactivada).

**Objetivo:** Reducir la carga administrativa y de formato, permitiendo al investigador centrarse en el contenido técnico.

## 📘 1. Explicación del Generador de Metaprompt

Este código automatiza la creación de instrucciones complejas para IAs, orientadas a la escritura técnica **IEEE**. Sus componentes principales son:

1.  **Entrada Dinámica (`input`)**: Captura el tema de investigación del usuario.
2.  **Plantilla de Ingeniería de Prompts (`Template`)**: Define un esquema estricto de tres partes (Estructura JSON, Bibliografía BibTeX 2023-2026 y Mapa de Uso) que fuerza a la IA a ser técnica y organizada.
3.  **Interfaz de Usuario (`ipywidgets`)**: Crea un área de texto para revisar el prompt generado.
4.  **Botón de Copiado Seguro (`JavaScript`)**: Implementa un script personalizado para evadir las restricciones de seguridad de Colab, permitiendo copiar el texto al portapapeles con un solo clic.

**Objetivo:** Obtener un 'plano' detallado del paper antes de empezar a redactar.

In [ ]:
# @title 🛠️ Generador de Metaprompt para Documentación IEEE
# @markdown Ingrese el tema o contenido de la investigación para procesar el metaprompt.

# 1. Captura de parámetros mediante input de texto
contenido_insertado = input("Ingrese el tema de investigación: ")

import json
import os
from string import Template
from ipywidgets import widgets
from IPython.display import display, HTML

# 2. Cargar la plantilla del Metaprompt desde el archivo markdown
metaprompt_template_path = 'templetes/templete_meta_prompt.md'

if not os.path.exists(metaprompt_template_path):
    print(f"Error: La plantilla no se encontró en {metaprompt_template_path}")
    metaprompt_content = ""
else:
    with open(metaprompt_template_path, 'r', encoding='utf-8') as f:
        metaprompt_content = f.read()

# Reemplazar {{contenido_insertado}} con $contenido para compatibilidad con string.Template
metaprompt_content = metaprompt_content.replace('{{contenido_insertado}}', '$contenido')
metaprompt_tpl = Template(metaprompt_content)

# 3. Sustitución de parámetros
prompt_final = metaprompt_tpl.substitute(contenido=contenido_insertado)

# 4. Interfaz de salida
print("\n✅ Prompt procesado con éxito.\n")

# Preparamos el string para JS de forma segura
prompt_json_esc = json.dumps(prompt_final)

# Botón para copiar con fallback a textarea para máxima compatibilidad en Colab
boton_copiar_html = HTML(f"""
    <script>
    function copiarAlPortapapeles() {{
        const text = {prompt_json_esc};
        const textArea = document.createElement("textarea");
        textArea.value = text;
        document.body.appendChild(textArea);
        textArea.select();
        try {{
            document.execCommand('copy');
            alert("Prompt copiado al portapapeles");
        }} catch (err) {{
            console.error('Error al copiar: ', err);
        }}
        document.body.removeChild(textArea);
    }}
    </script>
    <button onclick="copiarAlPortapapeles()"
    style="background-color: #4CAF50; color: white; padding: 10px 20px; border: none; border-radius: 4px; cursor: pointer; margin-bottom: 10px;">
    📋 Copiar Prompt al Portapapeles
    </button>
""")

output_area = widgets.Textarea(
    value=prompt_final,
    layout=widgets.Layout(width='98%', height='300px'),
    description='Prompt:',
    disabled=False
)

display(boton_copiar_html)
display(output_area)

## 📝 2. Gestor de Plan de Investigación

Este componente proporciona una interfaz interactiva para capturar y persistir la estructura detallada de la investigación. Sus funciones son:

1.  **Captura de Contenido**: Un área de texto de gran capacidad para pegar el plan en formato Markdown.
2.  **Persistencia Directa**: El botón **Guardar** crea automáticamente el directorio necesario y escribe el archivo en `workflow/workflow/research_plan.md`.
3.  **Control de Flujo**: Permite limpiar el área de trabajo rápidamente para nuevas iteraciones.
4.  **Validación**: Informa al usuario sobre el éxito de la operación y el tamaño del archivo guardado.

**Importancia:** Este archivo es el insumo principal para el procesador que genera el entorno LaTeX y los prompts de escritura.

In [ ]:
# @title
import os
from ipywidgets import widgets
from IPython.display import display

# Configuración de ruta
file_path = 'workflow/workflow/research_plan.md'

# 1. Crear widgets
text_area = widgets.Textarea(
    placeholder='Pegue aquí el contenido largo en Markdown...',
    description='Contenido:',
    layout=widgets.Layout(width='98%', height='400px')
)

save_button = widgets.Button(
    description='💾 Guardar Plan de Investigación',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

clear_button = widgets.Button(
    description='🗑️ Limpiar Área',
    button_style='warning',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()

# 2. Funciones de control
def on_save_clicked(b):
    with output:
        output.clear_output()
        try:
            # Esta línea crea las carpetas si no existen
            os.makedirs(os.path.dirname(file_path), exist_ok=True)
            # Al abrir en modo 'w', Python crea el archivo si no existe
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text_area.value)
            print(f"✅ Archivo guardado exitosamente en: {file_path}")
            print(f"📏 Tamaño: {len(text_area.value)} caracteres.")
        except Exception as e:
            print(f"❌ Error al guardar: {str(e)}")

def on_clear_clicked(b):
    text_area.value = ''
    with output:
        output.clear_output()
        print("🧹 Área de texto limpiada.")

save_button.on_click(on_save_clicked)
clear_button.on_click(on_clear_clicked)

# 3. Mostrar interfaz
print("📝 Gestor de Plan de Investigación")
display(text_area)
display(widgets.HBox([save_button, clear_button]))
display(output)

## ⚙️ 3. Procesador de Documentos (SRA 5.2)

Este script es el motor de automatización que transforma el plan de investigación en un entorno de trabajo LaTeX profesional. Sus funciones principales son:

1.  **Extracción de Bloques**: Utiliza Regex para identificar y separar la estructura JSON y los bloques BibTeX dentro del archivo Markdown.
2.  **Generación de Estructura IEEE**: Crea automáticamente el archivo `main.tex` con el preámbulo oficial de IEEE, metadatos del autor y configuración de `biblatex`.
3.  **Modularización**: Genera archivos independientes (`section_*.tex`) para cada parte del documento, facilitando la edición por separado.
4.  **Gestión Bibliográfica**: Consolida todas las fuentes en un archivo `references.bib` listo para ser procesado por Biber.

**Resultado:** Un entorno de desarrollo listo para cargar en Overleaf o compilar localmente con redacción asistida por IA.

In [ ]:
# @title
import json
import re
import os
import sys


def slugify(text):
    """Convierte el nombre del tema en un nombre de carpeta válido."""
    if not text:
        return "investigacion_nueva"
    text = text.lower().replace(" ", "_")
    return re.sub(r'(?u)[^-\w.]', '', text)


def extract_document_blocks(content):
    """
    Extrae todos los bloques de contenido relevantes del archivo.
    Retorna: (json_estructura_principal, dict_mapas_por_seccion, lista_bibtex)
    """
    json_blocks = re.findall(r'```json\s*(\{.*?\})\s*```', content, re.DOTALL)
    bib_blocks = re.findall(r'```bib(?:tex)?\s*(.*?)\s*```', content, re.DOTALL)

    if not json_blocks:
        return None, {}, bib_blocks

    estructura_principal = None
    mapas_por_seccion = {}

    for block in json_blocks:
        try:
            data = json.loads(block)
            if "titulo" in data and "secciones" in data:
                estructura_principal = data
            elif "mapa_uso" in data or "seccion_nro" in data:
                nro_seccion = data.get("seccion_nro", data.get("seccion", 0))
                mapas_por_seccion[nro_seccion] = data
        except json.JSONDecodeError:
            continue

    return estructura_principal, mapas_por_seccion, bib_blocks


def create_output_directory(input_file, project_name):
    input_dir = os.path.dirname(os.path.abspath(input_file))
    folder_name = slugify(project_name)
    output_path = os.path.join(input_dir, folder_name)

    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Carpeta creada exitosamente en: {output_path}")
    else:
        print(f"Usando carpeta existente: {output_path}")

    return output_path


def load_image_manifest(output_path):
    manifest_path = os.path.join(output_path, 'image_manifest.json')
    if os.path.exists(manifest_path):
        try:
            with open(manifest_path, 'r', encoding='utf-8') as mf:
                return json.load(mf)
        except json.JSONDecodeError:
            return {}
    return {}


def save_image_manifest(output_path, manifest_data):
    manifest_path = os.path.join(output_path, 'image_manifest.json')
    with open(manifest_path, 'w', encoding='utf-8') as mf:
        json.dump(manifest_data, mf, indent=4, ensure_ascii=False)


def write_section_tex_files(data, output_path):
    sections = data.get('secciones', [])
    created = []

    for sec in sections:
        n = sec.get('nro', 0)
        sec_title = sec.get('titulo_seccion', f'Seccion {n}')
        filename = os.path.join(output_path, f'section_{n}.tex')
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"% Contenido para la sección: {sec_title}\n")
            f.write(f"\\section{{{sec_title}}}\n")
        created.append(filename)

    print(f"✅ Generados {len(created)} archivos section_*.tex")
    return created


def build_citation_instructions(sec, mapas_referencias):
    n = sec.get('nro', 0)
    mapa_uso = mapas_referencias.get(n, {})
    mapa_detalle = mapa_uso.get('mapa_uso', {})
    instrucciones = []

    if mapa_detalle:
        instrucciones.append("\n📚 MAPA DE USO DE REFERENCIAS (OBLIGATORIO):")
        for key, info in mapa_detalle.items():
            razon = info.get('razon_seleccion', 'Sin justificación')
            guia = info.get('guia_redaccion', 'Integrar de forma natural')
            subseccion = info.get('subseccion_destino', 'Cualquiera')
            instrucciones.append(f"\n• **{key}** (Subsección {subseccion}):")
            instrucciones.append(f"  - Razón: {razon}")
            instrucciones.append(f"  - Instrucción: {guia}")

    return '\n'.join(instrucciones)


def generate_section_files(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    content = content.replace('\u00a0', ' ')
    data, mapas_referencias, bib_blocks = extract_document_blocks(content)

    if not data:
        print("Error: No se encontró el JSON de estructura principal (debe contener 'titulo' y 'secciones').")
        return

    if not project_name:
        project_name = data.get('titulo', 'investigacion_nueva')

    output_path = create_output_directory(input_file, project_name)
    write_section_tex_files(data, output_path)


def generate_research_files(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    content = content.replace('\u00a0', ' ')
    data, mapas_referencias, bib_blocks = extract_document_blocks(content)

    if not data:
        print("Error: No se encontró el JSON de estructura principal (debe contener 'titulo' y 'secciones').")
        return

    if not bib_blocks:
        print("Error: No se encontraron bloques BibTeX en el archivo.")
        return

    if not project_name:
        project_name = data.get('titulo', 'investigacion_nueva')

    output_path = create_output_directory(input_file, project_name)
    write_section_tex_files(data, output_path)

    full_bib_content = '\n\n'.join([b.strip() for b in bib_blocks])
    with open(os.path.join(output_path, 'references.bib'), 'w', encoding='utf-8') as f:
        f.write(full_bib_content)

    abstract_content = data.get('abstract_preliminar', 'Abstract no disponible.')
    with open(os.path.join(output_path, 'abstract.tex'), 'w', encoding='utf-8') as f:
        f.write('\\begin{abstract}\n' + abstract_content + '\n\\end{abstract}')

    sections = data.get('secciones', [])
    main_tex = r"""\documentclass[10pt, journal, final, twocolumn, letterpaper]{IEEEtran}

% --- PREÁMBULO DE PAQUETES ---
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage{amsmath, amssymb, amsfonts, amsthm}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{array}
\usepackage{url}
\usepackage{hyperref}
\usepackage{color, xcolor}
\usepackage{orcidlink} % Paquete específico para el icono y link de ORCID
\usepackage[backend=biber, style=ieee, natbib=true]{biblatex}

% --- CONFIGURACIÓN DE BIBLIOGRAFÍA ---
\addbibresource{references.bib}

% --- METADATOS DEL DOCUMENTO ---
\title{""" + data.get('titulo', project_name) + r"""

\author{
    \IEEEauthorblockN{Héctor A. Martínez \orcidlink{0009-0000-8039-3585}},
    Agencia Bolivariana para Actividades Espaciales\\
    Email: hmartinez@abae.gob.ve
}

\begin{document}
\maketitle

\input{abstract.tex}

"""

    for sec in sections:
        n = sec.get('nro', 0)
        main_tex += f"\\input{{section_{n}.tex}}\n"

    main_tex += "\n% --- BIBLIOGRAFÍA ---\n%\\nocite{*}\n" + "\\printbibliography" + "\n\\end{document}"
    with open(os.path.join(output_path, 'main.tex'), 'w', encoding='utf-8') as f:
        f.write(main_tex)

    print(f"✅ Éxito. Archivos generados en: {project_name}")
    print(f"📊 Mapas de uso detectados: {len(mapas_referencias)} secciones")
    if mapas_referencias:
        print(f"   Secciones con mapa: {sorted(mapas_referencias.keys())}")


if __name__ == "__main__":
    # Define the input file and mode
    input_file = 'workflow/workflow/research_plan.md'
    mode = 'all'

    # Check if the research plan exists before running
    if os.path.exists(input_file):
        print(f"🚀 Starting project generation for: {input_file}\n")
        # We call the main function from the previous cell directly for better integration in the notebook
        generate_research_files(input_file, 'outputs')
    else:
        print(f"❌ Error: The file {input_file} was not found. Please ensure you saved the research plan first.")


## 4. Redacción Asistida


### 🚀 4.1 Visualizador de Prompts por Sección

Este componente final permite interactuar con los resultados del procesador sin salir de la interfaz de Colab. Sus funciones clave son:

1.  **Exploración Granular**: Permite cargar individualmente el prompt de cualquier sección mediante su número identificador.
2.  **Copiado Seguro e Instantáneo**: Integra el mismo motor de JavaScript desarrollado para el Metaprompt, asegurando que el contenido técnico (usualmente muy largo) se copie correctamente al portapapeles.
3.  **Previsualización**: Ofrece un área de texto para validar el contenido antes de usarlo en el chat de redacción asistida.

**Flujo de Trabajo:** Ingrese el número de sección -> Cargar -> Copiar -> Pegar en su IA de redacción preferida.

In [ ]:
# @title Generador de Prompts de secciones
import os
import json
import re
from ipywidgets import widgets
from IPython.display import display, HTML

# Rutas de archivos
input_plan_path = 'workflow/workflow/research_plan.md'
template_path = 'templetes/section_meta_prompt.md' # Ajustado a la ruta común en Colab

def generate_prompt_on_the_fly(section_number):
    # 1. Validaciones de archivos
    if not os.path.exists(input_plan_path):
        return f"❌ Error: No se encontró el archivo del plan en {input_plan_path}."

    if not os.path.exists(template_path):
        return f"❌ Error: No se encontró la plantilla en {template_path}."

    # 2. Leer Plan de Investigación
    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')

    data, mapas_referencias, _ = extract_document_blocks(content)

    if not data:
        return "❌ Error: No se pudo extraer la estructura del plan."

    sections = data.get('secciones', [])
    sec = next((s for s in sections if s.get('nro') == section_number), None)

    if not sec:
        return f"❌ Error: No se encontró la sección {section_number} en el plan."

    # 3. Preparar datos para la plantilla
    sec_title = sec.get('titulo_seccion', f'Sección {section_number}')
    project_title = data.get('titulo', 'Investigación')
    instrucciones_citacion = build_citation_instructions(sec, mapas_referencias)

    if not instrucciones_citacion:
        instrucciones_citacion = "\n⚠️ ADVERTENCIA: No se encontró mapa de uso para esta sección. Integra las citas de forma coherente.\n"

    # Diccionario de mapeo entre etiquetas de plantilla y variables
    replacements = {
        "{{section_number}}": str(section_number),
        "{{project_title}}": project_title,
        "{{sec_title}}": sec_title,
        "{{objetivos}}": ", ".join(sec.get('objetivos', [])),
        "{{subsecciones}}": ", ".join(sec.get('subsecciones', [])),
        "{{insumos}}": ", ".join(sec.get('insumos', [])),
        "{{llaves_bibtex}}": ", ".join(sec.get('llaves_bibtex', [])),
        "{{instrucciones_citacion}}": instrucciones_citacion
    }

    # 4. Cargar plantilla y reemplazar
    with open(template_path, 'r', encoding='utf-8') as f:
        prompt_template = f.read()

    for key, value in replacements.items():
        prompt_template = prompt_template.replace(key, value)

    return prompt_template

# --- Interfaz de Usuario (Sin cambios significativos) ---
section_input = widgets.IntText(value=1, description='Sección:', layout=widgets.Layout(width='150px'))
btn_generate = widgets.Button(description='⚡ Generar desde Plantilla', button_style='primary')
out_area = widgets.Output()

def on_gen_clicked(b):
    with out_area:
        out_area.clear_output()
        p_content = generate_prompt_on_the_fly(section_input.value)

        if "❌" in p_content:
            print(p_content)
            return

        p_esc = json.dumps(p_content)
        copy_btn = HTML(f"""
            <script>
            function copyDynamic() {{
                const t = {p_esc};
                const el = document.createElement('textarea'); el.value = t;
                document.body.appendChild(el); el.select();
                document.execCommand('copy'); document.body.removeChild(el);
                alert('Prompt generado y copiado desde plantilla');
            }}
            </script>
            <button onclick="copyDynamic()" style="background:#28a745;color:white;padding:8px;border:none;border-radius:4px;cursor:pointer;">📋 Copiar Prompt</button>
        """)

        display(copy_btn)
        display(widgets.Textarea(value=p_content, layout=widgets.Layout(width='98%', height='350px')))

btn_generate.on_click(on_gen_clicked)
display(widgets.HBox([section_input, btn_generate]), out_area)

### 🚀 4.2 Procesador Directo: IA -> Archivos Finales

Este componente proporciona una interfaz interactiva para capturar la salida de una IA (que incluye bloques LaTeX y JSON) y procesarla directamente. Extrae el contenido LaTeX y lo guarda en `section_N.tex`, mientras que el JSON (con prompts de imagen) se fusiona y actualiza en `image_manifest.json`. Finalmente genera el `placeholder` de las imágenes de insumos previstas en la redacción y que después tendrán que ser generadas segun los prompts del `image_manifest.json`. Facilita la integración directa de la redacción asistida por IA en tu flujo de trabajo LaTeX.

In [ ]:
# @title
import os
import json
import re
from ipywidgets import widgets
from IPython.display import display
from PIL import Image, ImageDraw

# --- CONFIGURACIÓN DE RUTAS ---
BASE_PATH = 'workflow/workflow/outputs'
JSON_PATH = os.path.join(BASE_PATH, 'image_manifest.json')

# --- LÓGICA DE PLACEHOLDERS ---
def create_placeholders(project_folder):
    manifest_path = os.path.join(project_folder, 'image_manifest.json')
    if not os.path.exists(manifest_path):
        return "⚠️ No se encontró el manifiesto para generar placeholders."

    with open(manifest_path, 'r', encoding='utf-8') as f:
        try:
            manifest = json.load(f)
        except:
            return "❌ Error al leer el manifiesto."

    created_count = 0
    for filename, info in manifest.items():
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            filename += ".png"

        file_path = os.path.join(project_folder, filename)
        if os.path.exists(file_path):
            continue

        # Crear imagen gris
        img = Image.new('RGB', (800, 600), color=(220, 220, 220))
        d = ImageDraw.Draw(img)

        if isinstance(info, dict):
            seccion = info.get('seccion', 'N/A')
            desc = info.get('descripcion_original', 'Diagrama Técnico')
        else:
            seccion = "Procesando..."
            desc = "Revisar Prompt en JSON"

        text = f"PLACEHOLDER\nFILE: {filename}\nSEC: {seccion}\n{desc[:40]}..."
        d.text((40, 260), text, fill=(0, 0, 0))
        img.save(file_path)
        created_count += 1

    return f"🎨 Placeholders: {created_count} creados/actualizados."

# --- LÓGICA DE PROCESAMIENTO ---
def process_content(content, section_num):
    try:
        os.makedirs(BASE_PATH, exist_ok=True)
        target_tex_path = os.path.join(BASE_PATH, f"section_{section_num}.tex")

        latex_match = re.search(r'```latex\s*(.*?)\s*\n```', content, re.DOTALL)
        json_match = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)

        results = []

        if latex_match:
            latex_content = latex_match.group(1).strip()
            with open(target_tex_path, 'w', encoding='utf-8') as f:
                f.write(latex_content)
            results.append(f"✅ LaTeX inyectado en: {target_tex_path}")
        else:
            results.append("⚠️ No se encontró bloque LaTeX.")

        if json_match:
            new_prompts = json.loads(json_match.group(1).strip())
            existing_data = {}

            if os.path.exists(JSON_PATH):
                with open(JSON_PATH, 'r', encoding='utf-8') as f:
                    try:
                        existing_data = json.load(f)
                    except json.JSONDecodeError:
                        existing_data = {}

            for key, prompt in new_prompts.items():
                if key in existing_data and isinstance(existing_data[key], dict):
                    existing_data[key]["prompt_ia"] = prompt
                else:
                    existing_data[key] = {"prompt_ia": prompt}

            with open(JSON_PATH, 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, indent=4, ensure_ascii=False)
            results.append(f"✅ JSON actualizado en: {JSON_PATH}")

            # Generar placeholders automáticamente
            res_placeholder = create_placeholders(BASE_PATH)
            results.append(res_placeholder)
        else:
            results.append("⚠️ No se encontró bloque JSON.")

        return "\n".join(results)

    except Exception as e:
        return f"❌ Error crítico: {str(e)}"

# --- INTERFAZ DE USUARIO ---
print("🚀 Procesador Directo + Generador de Placeholders")
section_input = widgets.Text(value='1', description='Sección #:', layout=widgets.Layout(width='200px'))
text_capture = widgets.Textarea(placeholder='Pegue aquí el bloque completo de la IA...', description='Contenido:', layout=widgets.Layout(width='98%', height='350px'))
process_btn = widgets.Button(description='⚡ Procesar y Crear Imágenes', button_style='success', layout=widgets.Layout(width='250px', margin='10px 0px'))
status_out = widgets.Output()

def on_button_clicked(b):
    with status_out:
        status_out.clear_output()
        if not text_capture.value.strip(): return
        print("⏳ Procesando...")
        print(process_content(text_capture.value, section_input.value))

process_btn.on_click(on_button_clicked)
display(section_input, text_capture, process_btn, status_out)

## 📦 5. Compresor y Descargador de Resultados

Este código **comprime la carpeta de resultados `workflow/workflow/outputs/` (copiando el research_plan.md`!) en un archivo ZIP (`outputs.zip`) y lo descarga** a tu equipo local. Verifica la existencia de la carpeta antes de comprimir y proporciona un mensaje de error si no la encuentra. Es útil para empaquetar y transferir fácilmente todos los archivos generados desde Colab.

In [ ]:
# @title
import os
import shutil
from google.colab import files

# Direct path to the generated outputs
folder_to_zip = 'workflow/workflow/outputs/'
zip_filename = 'outputs.zip'

# Define the source and destination for the research plan copy
research_plan_source = 'workflow/workflow/research_plan.md'
research_plan_destination = os.path.join(folder_to_zip, os.path.basename(research_plan_source))

if os.path.exists(folder_to_zip):
    # Copy research_plan.md to the outputs folder before zipping
    if os.path.exists(research_plan_source):
        try:
            shutil.copy(research_plan_source, research_plan_destination)
            print(f"✅ Archivo '{research_plan_source}' copiado a '{research_plan_destination}'.")
        except Exception as e:
            print(f"❌ Error al copiar '{research_plan_source}': {e}")
    else:
        print(f"⚠️ Advertencia: El archivo '{research_plan_source}' no se encontró y no pudo ser copiado.")

    # Create zip from the outputs directory
    shutil.make_archive('outputs', 'zip', folder_to_zip)

    print(f"✅ Folder '{folder_to_zip}' compressed successfully.")
    files.download('outputs.zip')
else:
    print(f"❌ Error: Folder '{folder_to_zip}' not found. Please run the generation cell first.")

## 📝 6. Generador de Prompt de Registro

Este código genera un prompt estandarizado para registrar la investigación en un sistema de gestión documental externo.

**Funciones clave:**
1.  **Extracción**: Lee `research_plan.md` y extrae los datos técnicos (título, abstract, objetivos).
2.  **Construcción**: Crea un texto con campos obligatorios (Metodología, Alcance, Recursos, etc.) basados en el plan.
3.  **Interfaz**: Proporciona un botón para generar el prompt y otro para copiarlo al portapapeles mediante JavaScript, facilitando el pegado en herramientas de IA.

In [ ]:
# @title
import os
import json
from ipywidgets import widgets
from IPython.display import display, HTML

# Esta celda usa extract_document_blocks definida previamente
input_plan_path = 'workflow/workflow/research_plan.md'

def generate_register_prompt_on_the_fly():
    if not os.path.exists(input_plan_path):
        return f"❌ Error: No se encontró el archivo {input_plan_path}."

    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')

    data, _, _ = extract_document_blocks(content)

    if not data:
        return "❌ Error: No se pudo extraer la estructura del plan para el registro."

    project_title = data.get('titulo', 'Investigación Nueva')

    prompt = f'Necesito ingresar en el sistema de gestión documental de la Agencia los datos de la investigación: **{project_title}**.\n\n'
    prompt += 'Campos requeridos:\n'
    prompt += '* Title\n'
    prompt += '* Description\n'
    prompt += '* General Objective\n'
    prompt += '* Specific Objectives\n'
    prompt += '* Justification\n'
    prompt += '* Methodology\n'
    prompt += '* Scope (máximo 200 caracteres)\n'
    prompt += '* Activities\n'
    prompt += '* Resources\n'
    prompt += '* Limitations\n\n'
    prompt += 'Proporciona los valores en español basándote en el abstract y la estructura del paper.'

    return prompt

# Interfaz
btn_reg = widgets.Button(description='📝 Generar Prompt Registro', button_style='warning', layout=widgets.Layout(width='250px'))
reg_out = widgets.Output()

def on_reg_clicked(b):
    with reg_out:
        reg_out.clear_output()
        r_content = generate_register_prompt_on_the_fly()

        if "❌" in r_content:
            print(r_content)
            return

        r_esc = json.dumps(r_content)
        copy_btn_reg = HTML(f"""
            <script>
            function copyRegister() {{
                const t = {r_esc};
                const el = document.createElement('textarea'); el.value = t;
                document.body.appendChild(el); el.select();
                document.execCommand('copy'); document.body.removeChild(el);
                alert('Prompt de Registro copiado');
            }}
            </script>
            <button onclick="copyRegister()" style="background:#ff9800;color:white;padding:8px;border:none;border-radius:4px;cursor:pointer;margin-bottom:10px;">📋 Copiar Prompt de Registro</button>
        """)

        display(copy_btn_reg)
        display(widgets.Textarea(value=r_content, layout=widgets.Layout(width='98%', height='300px')))

btn_reg.on_click(on_reg_clicked)
display(btn_reg, reg_out)

## 🧹 Limpieza de Espacio de Trabajo

Este comando (actualmente desactivado) permite **eliminar permanentemente** la carpeta de resultados `outputs/`. Se utiliza para reiniciar el proceso de generación o limpiar archivos residuales antes de una nueva ejecución.

In [ ]:
#rm -r workflow/ outputs.zip